# Tanager Mangrove Mapping - 02 Classification

| | |
|---|---|
| **Authors**     | Muhammad Wahyu Ramadhan, Athar Abdurrahman B., Diniyarti |
| **Competition** | Planet Tanager Open Data Competition 2026 |
| **Topic**       | Transferable Mangrove Extent and Biomass Mapping Using Adaptive Spectral Thresholds |
| **Date**        | June 2026 |

---

**Scope:** Pseudo-label generation from adaptive thresholds, RF + XGBoost training on Sangatta, accuracy evaluation, wall-to-wall extent map.

## 0. Environment Setup

In [ ]:
%%javascript
function KeepAlive() {
  document.querySelector("#top-toolbar").click();
  console.log("keep alive: " + new Date().toLocaleTimeString());
}
setInterval(KeepAlive, 60000);  // klik setiap 60 detik

In [ ]:
# Install dependencies (Colab)
!pip install scikit-learn xgboost geopandas rasterio matplotlib joblib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio

# ============================================================
# Project root (adjust per environment)
# ============================================================
# Google Colab (Google Drive mounted)
ROOT = Path('/content/drive/MyDrive/PROJECT/Planet Tanager Competition 2026/tanager-mangrove-mapping')

# Local (uncomment if running locally)
# ROOT = Path('..').resolve()

DATA_PROC   = ROOT / 'data' / 'processed'
DATA_GMW    = ROOT / 'data' / 'gmw_v3'
OUT_MODELS  = ROOT / 'outputs' / 'models'
OUT_RESULTS = ROOT / 'outputs' / 'results'
OUT_FIGURES = ROOT / 'outputs' / 'figures'

# Ensure output folders exist
for d in (OUT_MODELS, OUT_RESULTS, OUT_FIGURES):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT))
from src.preprocessing import load_geotiff_bands, compute_all_indices
from src.classification import (
    generate_pseudo_labels,
    build_feature_matrix,
    split_data,
    train_random_forest,
    train_xgboost,
    tune_random_forest,
    evaluate_model,
    compare_models,
    predict_extent,
    save_model
)

# ============================================================
# Scene IDs (same dict as 01_preprocessing.ipynb)
# ============================================================
SCENES = {
    'sangatta'   : '20250302_030003_92_4001',
    'gujarat'    : '20250311_061550_53_4001',
    'elsalvador' : '20250223_165546_32_4001',
    'belize'     : '20250824_171857_84_4001',
    'australia'  : '20250608_014315_58_4001',
}

SITE     = 'sangatta'
SCENE_ID = SCENES[SITE]

print(f'ROOT     : {ROOT}')
print(f'SITE     : {SITE}')
print(f'SCENE_ID : {SCENE_ID}')

In [ ]:
# ============================================================
# Reload src modules during development
# ============================================================
import importlib
import src.preprocessing  as _pre
import src.classification as _cls
import src.evaluation     as _eval
import src.continuum      as _cont
importlib.reload(_pre)
importlib.reload(_cls)
importlib.reload(_eval)
importlib.reload(_cont)
from src.preprocessing import load_geotiff_bands, compute_all_indices
from src.classification import (
    generate_pseudo_labels, build_feature_matrix, split_data,
    train_random_forest, train_xgboost, evaluate_model,
    compare_models, predict_extent, save_model, load_model,
    tune_random_forest, tune_xgboost,
)
from src.evaluation import (
    rasterize_gmw, evaluate_against_gmw,
    plot_confusion_matrix, plot_agreement_map,
)
print('Reloaded.')

## 1. Load Indices + Thresholds

In [ ]:
# ============================================================
# Load multiband GeoTIFF (produced in 01) and recompute indices
# ============================================================
data    = load_geotiff_bands(str(DATA_PROC), SCENE_ID, site=SITE)
indices = compute_all_indices(data)

print('Index statistics:')
for name, arr in indices.items():
    print(f'  {name:<8}: min={np.nanmin(arr):.3f}  max={np.nanmax(arr):.3f}  mean={np.nanmean(arr):.3f}')

In [ ]:
# ============================================================
# Load thresholds saved in 01_preprocessing.ipynb
# ============================================================
thresh_path = OUT_RESULTS / f'thresholds_{SITE}_{SCENE_ID}.json'
with open(thresh_path) as f:
    thresholds = json.load(f)

print('Thresholds loaded:')
for k, v in thresholds.items():
    print(f'  {k:<8}: {v:.4f}')

In [ ]:
# ============================================================
# Load coastal candidate mask (from 01_preprocessing)
# ============================================================
cand_path = DATA_PROC / f'candidate_{SITE}_{SCENE_ID}.tif'
with rasterio.open(cand_path) as src:
    candidate_mask = src.read(1).astype(bool)
print(f'Candidate mask loaded : {candidate_mask.sum():,} px ({100*candidate_mask.mean():.1f}%)')

# ============================================================
# Load hyperspectral diagnostic feature REIP (from 01_preprocessing)
# Saved on HDF5 grid (850x810 for Sangatta) -- must resample to
# GeoTIFF/indices grid (667x915) before use in classifier.
# Shape mismatch arises because hdf5_to_geotiff() reprojects to a
# slightly different pixel grid than the original HDF5.
# ============================================================
from rasterio.warp import reproject, Resampling

def resample_to_grid(src_array, src_transform, src_crs,
                     dst_shape, dst_transform, dst_crs):
    """Resample 2D float32 array from source grid to destination grid."""
    dst = np.full(dst_shape, np.nan, dtype=np.float32)
    reproject(
        source=src_array,
        destination=dst,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear,
        src_nodata=np.nan,
        dst_nodata=np.nan,
    )
    return dst

# Target grid: same as spectral indices
dst_shape     = indices['NDMI'].shape
dst_transform = data['transform']
dst_crs       = data['crs']

reip_path = DATA_PROC / f'reip_{SITE}_{SCENE_ID}.tif'

with rasterio.open(reip_path) as src:
    reip_map_r = resample_to_grid(
        src.read(1), src.transform, src.crs,
        dst_shape, dst_transform, dst_crs
    )

extra_features = {
    'REIP' : reip_map_r,
}
print(f'REIP resampled : {np.isfinite(reip_map_r).sum():,} valid px, mean={np.nanmean(reip_map_r):.1f} nm')

## 2. Pseudo-label Generation

In [ ]:
# ============================================================
# AND logic: MVI > threshold AND NDMI > threshold
# Spatially constrained to coastal candidate zone
# ============================================================
labels = generate_pseudo_labels(indices, thresholds,
                                 candidate_mask=candidate_mask)

## 3. Feature Matrix + Train/Test Split

In [ ]:
X, y, feature_names = build_feature_matrix(
    indices, labels, extra_features=extra_features
)
X_train, X_test, y_train, y_test = split_data(X, y)

In [ ]:
# ============================================================
# Pseudo-label map + train/test spatial distribution
# ============================================================
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Panel 1: Pseudo-label raster map ---
label_vis = labels.astype(float)
label_vis[labels == -1] = np.nan

cmap_label = ListedColormap(['#d9d9d9', '#2ca02c'])
axes[0].imshow(label_vis, cmap=cmap_label, vmin=0, vmax=1, interpolation='none')
axes[0].set_title(f'Pseudo-label Map -- {SITE}\n'
                  f'mangrove={int((labels==1).sum()):,} px  '
                  f'non-mangrove={int((labels==0).sum()):,} px')
axes[0].axis('off')
legend_0 = [
    Patch(color='#2ca02c', label=f'Mangrove ({int((labels==1).sum()):,} px)'),
    Patch(color='#d9d9d9', label=f'Non-mangrove ({int((labels==0).sum()):,} px)'),
]
axes[0].legend(handles=legend_0, loc='lower right', fontsize=8)

# --- Panel 2: Train/Test spatial distribution ---
valid_mask_2d = labels != -1
rows, cols    = np.where(valid_mask_2d)
n_valid       = len(rows)

# Reconstruct same split as training (identical random_state)
idx_all              = np.arange(n_valid)
idx_train, idx_test  = train_test_split(
    idx_all, test_size=0.2, stratify=y, random_state=42
)

# Background: MVI for spatial context
axes[1].imshow(indices['MVI'], cmap='Greys', vmin=0, vmax=5, alpha=0.4)

# Subsample for display (scatter too slow at 200k+ pts)
MAX_PTS = 5000
rng     = np.random.RandomState(0)

for idx_set, color, label_txt in [
    (idx_train, '#1f77b4', f'Train ({len(idx_train):,})'),
    (idx_test,  '#ff7f0e', f'Test  ({len(idx_test):,})'),
]:
    sub = idx_set if len(idx_set) <= MAX_PTS else \
          rng.choice(idx_set, MAX_PTS, replace=False)
    axes[1].scatter(cols[sub], rows[sub],
                    c=color, s=1, alpha=0.5, linewidths=0, label=label_txt)

axes[1].set_title(f'Train/Test Spatial Distribution -- {SITE}\n'
                  f'(subsampled to {MAX_PTS:,} pts/set for display)')
axes[1].axis('off')
axes[1].legend(loc='lower right', fontsize=8, markerscale=5)

plt.tight_layout()
plt.savefig(OUT_FIGURES / f'pseudo_label_map_{SITE}_{SCENE_ID}.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Zoom-in panels: 2 random coastal locations
# ============================================================
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 7))

# Two random coastal zoom windows (row, col center) -- within scene bounds
# Chosen to capture coastal strip where mangrove pseudo-labels exist
H_scene, W_scene = indices['MVI'].shape
ZOOM = 80   # half-window size in pixels (~2.4km at 30m)

zoom_centers = [
    (180, 530),   # northern coastal strip
    (420, 480),   # central/southern coastal area
]
zoom_titles = ['Coastal Zoom -- North', 'Coastal Zoom -- Central/South']

for ax, (cr, cc), title in zip(axes2, zoom_centers, zoom_titles):
    r0 = max(cr - ZOOM, 0);  r1 = min(cr + ZOOM, H_scene)
    c0 = max(cc - ZOOM, 0);  c1 = min(cc + ZOOM, W_scene)

    # Background: MVI
    ax.imshow(indices['MVI'][r0:r1, c0:c1],
              cmap='Greys', vmin=0, vmax=5, alpha=0.5,
              extent=[c0, c1, r1, r0])

    # Pseudo-label overlay (mangrove only, for reference)
    label_zoom = labels[r0:r1, c0:c1].astype(float)
    label_zoom[label_zoom != 1] = np.nan
    ax.imshow(label_zoom, cmap=ListedColormap(['#2ca02c']),
              vmin=0, vmax=1, alpha=0.4,
              extent=[c0, c1, r1, r0], interpolation='none')

    # Subsample within window (max 500 pts per class for zoom clarity)
    MAX_ZOOM = 500
    rng_zoom = np.random.RandomState(42)

    # Scatter train/test points within window
    for idx_set, color, label_txt, marker in [
        (idx_train, '#1f77b4', 'Train', 'o'),
        (idx_test,  '#ff7f0e', 'Test',  's'),
    ]:
        in_window = (
            (rows[idx_set] >= r0) & (rows[idx_set] < r1) &
            (cols[idx_set] >= c0) & (cols[idx_set] < c1)
        )
        idx_win = np.where(in_window)[0]
        if len(idx_win) > MAX_ZOOM:
            idx_win = rng_zoom.choice(idx_win, MAX_ZOOM, replace=False)
        r_win = rows[idx_set[idx_win]]
        c_win = cols[idx_set[idx_win]]
        ax.scatter(c_win, r_win, c=color, s=20, alpha=0.8,
                   linewidths=0.3, edgecolors='white',
                   label=f'{label_txt} ({in_window.sum():,})',
                   marker=marker)

    ax.set_xlim(c0, c1)
    ax.set_ylim(r1, r0)
    ax.set_title(f'{title}\n(~{(r1-r0)*30/1000:.1f} x {(c1-c0)*30/1000:.1f} km)')
    ax.axis('off')
    ax.legend(loc='lower right', fontsize=8, markerscale=2)

plt.suptitle(f'Train/Test Distribution -- Coastal Zoom -- {SITE}', y=1.01)
plt.tight_layout()
plt.savefig(OUT_FIGURES / f'pseudo_label_zoom_{SITE}_{SCENE_ID}.png',
            dpi=150, bbox_inches='tight')
plt.show()

## 4. Model Training

In [ ]:
# ============================================================
# Random Forest (primary model)
# ============================================================
print('Training Random Forest...')
rf_model = train_random_forest(X_train, y_train)
save_model(rf_model, str(OUT_MODELS / f'rf_{SITE}_{SCENE_ID}.joblib'))

In [ ]:
# ============================================================
# XGBoost (comparison model)
# ============================================================
print('Training XGBoost...')
xgb_model = train_xgboost(X_train, y_train)
save_model(xgb_model, str(OUT_MODELS / f'xgb_{SITE}_{SCENE_ID}.joblib'))

### 4c. RF Hyperparameter Tuning (RandomizedSearchCV, CV=5)

Tune RF hyperparameters on the pseudo-label training set. Scoring: F1 (mangrove class). Final evaluation against GMW v3 in Section 8.


In [ ]:
# ============================================================
# Hyperparameter tuning -- RF (primary) + XGBoost (comparison)
# RandomizedSearchCV, n_iter=50, cv=5, scoring=F1 mangrove
# Gold-standard search space (scipy.stats distributions)
# ============================================================
import json as _json
import time

RUN_RF  = False  # set True/False
RUN_XGB = True

# ---- Random Forest ----
if RUN_RF:
    print("=" * 50)
    print("  Tuning: Random Forest")
    print("=" * 50)
    t0 = time.time()
    rf_tuned, rf_best_params, rf_best_cv_f1 = tune_random_forest(
        X_train, y_train, n_iter=50, cv=5
    )
    rf_elapsed = time.time() - t0
    print(f'  Tuning duration : {rf_elapsed/60:.1f} min ({rf_elapsed:.0f} s)')
    rf_params_path = OUT_RESULTS / f'rf_best_params_{SITE}_{SCENE_ID}.json'
    with open(rf_params_path, 'w') as f:
        _json.dump({k: str(v) for k, v in rf_best_params.items()}, f, indent=2)
    print(f'  Params saved    : {rf_params_path.name}')
    save_model(rf_tuned, str(OUT_MODELS / f'rf_tuned_{SITE}_{SCENE_ID}.joblib'))
else:
    rf_tuned = load_model(str(OUT_MODELS / f'rf_tuned_{SITE}_{SCENE_ID}.joblib'))
    print("  RF tuned loaded from disk (RUN_RF=False)")

# ---- XGBoost ----
if RUN_XGB:
    print()
    print("=" * 50)
    print("  Tuning: XGBoost")
    print("=" * 50)
    t0 = time.time()
    xgb_tuned, xgb_best_params, xgb_best_cv_f1, xgb_search = tune_xgboost(
        X_train, y_train, n_iter=50, cv=5
    )
    xgb_elapsed = time.time() - t0
    print(f'  Tuning duration : {xgb_elapsed/60:.1f} min ({xgb_elapsed:.0f} s)')
    xgb_params_path = OUT_RESULTS / f'xgb_best_params_{SITE}_{SCENE_ID}.json'
    with open(xgb_params_path, 'w') as f:
        _json.dump({k: str(v) for k, v in xgb_best_params.items()}, f, indent=2)
    print(f'  Params saved    : {xgb_params_path.name}')
    save_model(xgb_tuned, str(OUT_MODELS / f'xgb_tuned_{SITE}_{SCENE_ID}.joblib'))
else:
    xgb_tuned = load_model(str(OUT_MODELS / f'xgb_tuned_{SITE}_{SCENE_ID}.joblib'))
    print("  XGB tuned loaded from disk (RUN_XGB=False)")

In [ ]:
# ============================================================
# CV score progression -- XGBoost
# ============================================================
if RUN_XGB:
    scores = xgb_search.cv_results_['mean_test_score']
    plt.figure(figsize=(8, 4))
    plt.plot(scores, marker='o', markersize=4)
    plt.axhline(xgb_best_cv_f1, color='red', linestyle='--', label=f'Best = {xgb_best_cv_f1:.4f}')
    plt.xlabel('Iteration'); plt.ylabel('Mean CV F1 (mangrove)')
    plt.title(f'RandomizedSearchCV progression -- XGBoost ({SITE})')
    plt.legend(); plt.tight_layout()
    plt.savefig(OUT_FIGURES / f'xgb_search_progression_{SITE}_{SCENE_ID}.png', dpi=150)
    plt.show()

## 5. Evaluation

In [ ]:
rf_metrics  = evaluate_model(rf_model,  X_test, y_test, 'Random Forest')
xgb_metrics = evaluate_model(xgb_model, X_test, y_test, 'XGBoost')

In [ ]:
# ============================================================
# Comparison table (saved to outputs/results/)
# ============================================================
comparison = compare_models(rf_metrics, xgb_metrics, y_test)
comparison.to_csv(OUT_RESULTS / f'accuracy_comparison_{SITE}_{SCENE_ID}.csv', index=False)
print(comparison)

## 6. Wall-to-Wall Extent Map

In [ ]:
# ============================================================
# Predict full scene using RF (primary model)
# Apply same spatial constraint as training (candidate zone only)
# ============================================================
h, w        = list(indices.values())[0].shape
extent_map  = predict_extent(rf_model, indices,
                              original_shape=(h, w),
                              candidate_mask=candidate_mask,
                              extra_features=extra_features)

# Save as GeoTIFF
extent_path = DATA_PROC / f'extent_mangrove_{SITE}_{SCENE_ID}.tif'
with rasterio.open(
    extent_path, 'w',
    driver='GTiff', height=h, width=w,
    count=1, dtype='int8',
    crs=data['crs'], transform=data['transform'],
    compress='lzw'
) as dst:
    dst.write(extent_map, 1)

print(f'Extent map saved : {extent_path}')

## 7. Visualization

In [ ]:
# ============================================================
# Extent map + GMW v3 overlay
# Adjust path if GMW v3 is stored elsewhere
# ============================================================
gmw_path = DATA_GMW / f'gmw_{SITE}_{SCENE_ID}.geojson'
gmw = gpd.read_file(gmw_path) if gmw_path.exists() else None

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# RF extent
axes[0].imshow(extent_map == 1, cmap='Greens')
axes[0].set_title(f'RF Mangrove Extent ({SITE})')
axes[0].axis('off')

# GMW v3 reference
if gmw is not None:
    gmw.plot(ax=axes[1], color='green', alpha=0.7)
    axes[1].set_title(f'GMW v3 Reference ({SITE})')
else:
    axes[1].text(0.5, 0.5, 'GMW v3 not found', ha='center', va='center')
    axes[1].set_title('GMW v3 (missing)')
axes[1].axis('off')

plt.tight_layout()
plt.savefig(OUT_FIGURES / f'extent_map_{SITE}_{SCENE_ID}.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Evaluation vs GMW v3 (Independent Ground Truth)

Section 5 evaluated against the pseudo-label test split, which is circular
(MVI/NDMI are both feature and label source). This section compares the
predicted extent against GMW v3, an independent reference never used in
training. These are the real accuracy figures.

In [ ]:
# ============================================================
# Rasterize GMW v3 polygons to match the extent_map grid
# ============================================================
gmw_path = DATA_GMW / f'gmw_{SITE}_{SCENE_ID}.geojson'

gmw_raster = rasterize_gmw(
    str(gmw_path),
    reference_shape=extent_map.shape,
    transform=data['transform'],
    crs=data['crs'],
)

In [ ]:
# ============================================================
# Evaluate RF extent vs GMW v3
# eval_mask = candidate_mask -> fair comparison (only where model
# was allowed to predict mangrove)
# ============================================================
gmw_eval = evaluate_against_gmw(
    extent_map,
    gmw_raster,
    eval_mask=candidate_mask,
    model_name='Random Forest',
)

# Save metrics
gmw_eval['metrics_table'].to_csv(
    OUT_RESULTS / f'gmw_eval_{SITE}_{SCENE_ID}.csv', index=False
)

In [ ]:
# ============================================================
# Evaluate RF tuned + XGBoost tuned vs GMW v3
# Comparison: Default RF | RF Tuned | XGBoost Tuned
# ============================================================
import time
import pandas as pd

h, w = list(indices.values())[0].shape

# RF tuned
t0 = time.time()
extent_rf_tuned = predict_extent(
    rf_tuned, indices,
    original_shape=(h, w),
    candidate_mask=candidate_mask,
    extra_features=extra_features,
)
print(f'RF tuned prediction : {time.time()-t0:.1f} s')
gmw_eval_rf_tuned = evaluate_against_gmw(
    extent_rf_tuned, gmw_raster,
    eval_mask=candidate_mask,
    model_name='RF Tuned',
)

# XGBoost tuned
t0 = time.time()
extent_xgb_tuned = predict_extent(
    xgb_tuned, indices,
    original_shape=(h, w),
    candidate_mask=candidate_mask,
    extra_features=extra_features,
)
print(f'XGB tuned prediction: {time.time()-t0:.1f} s')
gmw_eval_xgb_tuned = evaluate_against_gmw(
    extent_xgb_tuned, gmw_raster,
    eval_mask=candidate_mask,
    model_name='XGBoost Tuned',
)

# Comparison table: Default RF | RF Tuned | XGBoost Tuned
rows = [
    {'Model': 'RF Default',    'Kappa': gmw_eval['kappa'],
     'Precision': gmw_eval['precision'], 'Recall': gmw_eval['recall'],
     'F1': gmw_eval['f1_mangrove'], 'IoU': gmw_eval['IoU']},
    {'Model': 'RF Tuned',      'Kappa': gmw_eval_rf_tuned['kappa'],
     'Precision': gmw_eval_rf_tuned['precision'], 'Recall': gmw_eval_rf_tuned['recall'],
     'F1': gmw_eval_rf_tuned['f1_mangrove'], 'IoU': gmw_eval_rf_tuned['IoU']},
    {'Model': 'XGBoost Tuned', 'Kappa': gmw_eval_xgb_tuned['kappa'],
     'Precision': gmw_eval_xgb_tuned['precision'], 'Recall': gmw_eval_xgb_tuned['recall'],
     'F1': gmw_eval_xgb_tuned['f1_mangrove'], 'IoU': gmw_eval_xgb_tuned['IoU']},
]
df_compare = pd.DataFrame(rows)
print("\n  Model comparison vs GMW v3:")
print(df_compare.to_string(index=False))
df_compare.to_csv(
    OUT_RESULTS / f'model_comparison_{SITE}_{SCENE_ID}.csv', index=False
)

# Confusion + agreement maps for both tuned models
for model_name, extent_map_tuned, tag in [
    ('RF Tuned',      extent_rf_tuned,  'rf_tuned'),
    ('XGBoost Tuned', extent_xgb_tuned, 'xgb_tuned'),
]:
    plot_confusion_matrix(
        evaluate_against_gmw(extent_map_tuned, gmw_raster,
                             eval_mask=candidate_mask,
                             model_name=model_name)['confusion_matrix'],
        model_name=f'{model_name} vs GMW v3',
        normalize=True,
        save_path=str(OUT_FIGURES / f'confusion_{tag}_{SITE}_{SCENE_ID}.png'),
    )
    plot_agreement_map(
        extent_map_tuned, gmw_raster,
        eval_mask=candidate_mask,
        site=f'{SITE} ({model_name})',
        save_path=str(OUT_FIGURES / f'agreement_{tag}_{SITE}_{SCENE_ID}.png'),
    )


In [ ]:
# ============================================================
# Confusion matrix vs GMW v3
# ============================================================
plot_confusion_matrix(
    gmw_eval['confusion_matrix'],
    model_name='RF vs GMW v3',
    normalize=False,
    save_path=str(OUT_FIGURES / f'confusion_gmw_{SITE}_{SCENE_ID}.png'),
)

# Row-normalized version (shows recall per class)
plot_confusion_matrix(
    gmw_eval['confusion_matrix'],
    model_name='RF vs GMW v3 (normalized)',
    normalize=True,
    save_path=str(OUT_FIGURES / f'confusion_gmw_norm_{SITE}_{SCENE_ID}.png'),
)

In [ ]:
# ============================================================
# Spatial agreement map: where model agrees / over-predicts / misses
# ============================================================
plot_agreement_map(
    extent_map,
    gmw_raster,
    eval_mask=candidate_mask,
    site=SITE,
    save_path=str(OUT_FIGURES / f'agreement_{SITE}_{SCENE_ID}.png'),
)
